## Solving Cart Pole with Reinforce + Monte Carlo Policy Gradients

![Alt text](best_run.png)


In [5]:
import os

import gymnasium as gym
import structlog

# from samsara_rl.mdp.terminal_penalty_wrapper import TerminalPenaltyWrapper
from samsara_rl.control.function_approximation.batch.monte_carlo_policy_gradient.monte_carlo_policy_gradient import (
    MonteCarloPolicyGradient,
)
from samsara_rl.control.function_approximation.functions.neural_networks.fully_connected import (
    FullyConnected,
)

from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

In [6]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/reinforce_cart_pole_example"
ALPHAS = [0.0001, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [10]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single configuration.

    Creates a LinearFunction and QLearningGradient agent, trains for
    MAX_EPISODES episodes, saves the learned weights, and runs greedy
    evaluation.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the Q-learning update.
    """
    model_name = "reinforce"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    fc = FullyConnected(4, 32, 2, preprocess=None)

    agent = MonteCarloPolicyGradient(
        mdp=env, gamma=gamma, policy_network=fc, alpha=alpha, log_dir=run_dir, experiment_name=run_name
    )

    #     agent.register(CartPoleLogger())

    agent.evaluate(max_iter=MAX_EPISODES)
    #     save_model(agent, run_dir)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [11]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [12]:
main()

2026-09-17 18:03:27 [info     ] starting_run                   run='reinforce_alpha=0.0001_gamma=0.99'


/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


2026-09-17 18:04:40 [info     ] starting_run                   run='reinforce_alpha=0.0003_gamma=0.99'
2026-09-17 18:11:17 [info     ] starting_run                   run='reinforce_alpha=0.001_gamma=0.99'
